<a href="https://colab.research.google.com/github/SalesforceAIResearch/SlackAgents/blob/main/docs/docs/examples/tools/tool_definition_from_function.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Tools from Python Functions
This tutorial demonstrates how to use the `FunctionTool.from_function` method to create a function tool from a Python function.

If you're opening this Notebook on colab, you will probably need to install `SlackAgents`. You can do this by running the following cell:

```python
!pip install slackagents
```


In [ ]:
!pip install slackagents

## Import necessary modules


In [1]:
from slackagents.tools.function_tool import FunctionTool
from slackagents.tools.base import ToolCall

## Define a sample function
Now, let's define a sample function that we'll use to create our FunctionTool. Note that users are highly recommended to use type hints, together with standard Python docstrings, to provide information about the function's input and output. We currently support parsing of ReST, Google, Numpydoc-style and Epydoc docstrings.

Note that when defining a parameter type, we should strictly use JSON schema types. For example, use `string` instead of `str`, `number` instead of `float` or `int`, `boolean` instead of `bool`, and `array` instead of `list`. The FunctionTool will first try to use the type annotation in the function argument to automatically infer the type of the parameter. If no type annotation is provided, the FunctionTool will use the docstring type name, the worst case scenario, it's a string. For example, the following function, even if `:type length` is missing in the docstring, the FunctionTool will be able to infer it as `number` type from the type hint in the function argument. However, we still recommend to explicitly specify the type in the docstring for clarity.

```python

In [9]:
def calculate_area(length: float, width: float) -> float:
    """
    Calculate the area of a rectangle.
    
    :param length: The length of the rectangle.
    :type length: number
    :param width: The width of the rectangle.
    :type width: number
    :return: The area of the rectangle.
    """
    return length * width

## Create a FunctionTool using from_function
We can now create a FunctionTool using the `from_function` class method.

In [10]:
area_tool = FunctionTool.from_function(calculate_area)

The docstring and signature of the function are automatically parsed and stored in the tool's info attribute. The tool's name is set to the function's name by default. Let's look at the information about our newly created tool:

In [11]:
import json
print(json.dumps(area_tool.info, indent=4))

{
    "type": "function",
    "function": {
        "name": "calculate_area",
        "description": "Calculate the area of a rectangle.",
        "parameters": {
            "type": "object",
            "properties": {
                "length": {
                    "type": "number",
                    "description": "The length of the rectangle."
                },
                "width": {
                    "type": "number",
                    "description": "The width of the rectangle."
                }
            },
            "required": [
                "length",
                "width"
            ],
            "additionalProperties": false
        }
    }
}


## Execute the tool
To execute the tool, we need to create a ToolCall object and then use the execute method:

In [6]:
call = ToolCall(name="calculate_area", arguments={"length": 5, "width": 3})

In [7]:
result = area_tool.execute(call)
print(f"The area is: {result}")

The area is: 15.0


Note that our tool execution automatically casts the input in `ToolCall` to the correct types, as specified in the function signature. The output is also automatically cast to string, for appending to the multi-turn conversation.

In [9]:
# We on purpose pass a string instead of a number for the length
call = ToolCall(name="calculate_area", arguments={"length": "5", "width": 3})
result = area_tool.execute(call)
print(f"The area is: {result}")

The area is: 15.0


## Automatic error handling
Let's see what happens when we provide invalid arguments:

In [7]:
invalid_call = ToolCall(name="calculate_area", arguments={"length": 5})
error_result = area_tool.execute(invalid_call)
print(f"Error: {error_result}")

Error: {'error': "calculate_area() missing 1 required positional argument: 'width'"}


## Conclusion

This tutorial demonstrated how to use `FunctionTool.from_function` to create a tool from a Python function, examine its information, and execute it with both valid and invalid inputs.